# Day 34/100 — RipenessMeter V3 FINAL | Dual dataset, conservative leakage audit

**Dataset** `/kaggle/input/datasets/loliwibu/ripenessmeter-dataset`  
**Primary** BananaImageBD 820 source images, four native stages; **secondary** AnnotatedBanana 673 source images with detection boxes.  
**Supervised target** unripe / ripe / overripe, with explicit Green + Semi-ripe -> unripe operational mapping. Boxes become labeled object crops; never claim a whole-image detector.  

**V3 corrections:** source-native conflict quarantine; exact SHA-256 + conservative perceptual near-duplicate cross-split screening; ambiguous cross-split samples are quarantined and logged **before training**; two-domain selection metrics and held-out reports; uncertainty intervals, artifact integrity manifest, original-image lineage. Perceptual screening cannot certify biological subject independence: candidate flags require human review, and all quarantines are reported rather than silently relabeled.  

**Baseline:** previous V2 checkpoint remains separate; no fake claims that V3 outperforms V2 before executing this notebook. Training and output artifacts use unique V3 names. Source licenses and rights must be checked prior to redistribution. GPU T4; Internet On for ImageNet weights.


## 0 · Setup (T4, Internet ON for ImageNet weights)

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from io import BytesIO
import os, sys, json, random, hashlib, time, csv, zipfile, traceback, gc
import numpy as np, pandas as pd
from PIL import Image, ImageOps, ImageFile
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, balanced_accuracy_score

SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
INPUT=Path('/kaggle/input/datasets/loliwibu/ripenessmeter-dataset')
WORK=Path('/kaggle/working/RipenessMeter_Day34_V3_FINAL')
REPORTS=WORK/'reports';MODELS=WORK/'models';PLOTS=WORK/'assets/plots';EXAMPLES=WORK/'assets/examples';CACHE=WORK/'crop_cache'
for p in [REPORTS,MODELS,PLOTS,EXAMPLES,CACHE]:p.mkdir(parents=True,exist_ok=True)
CLASSES=['unripe','ripe','overripe'];CLASS_TO_ID={v:i for i,v in enumerate(CLASSES)}
FOUR_TO_THREE={'Green':'unripe','Semi-ripe':'unripe','Ripe':'ripe','Overripe':'overripe'}
YOLO_TO_CLASS={0:'unripe',1:'ripe',2:'overripe'}
DEVICE=torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
IMG_SIZE=224;BATCH=32;MAX_EPOCHS=35;PATIENCE=7;LR=2e-4;NUM_WORKERS=0
MAX_MINUTES=90;MIN_BOX_SIDE=12;MAX_CROPS_PER_IMAGE=12
assert INPUT.is_dir(),f'Input not mounted: {INPUT}'
assert DEVICE.type=='cuda','Enable Kaggle GPU accelerator before full training'
print('Python',sys.version.split()[0],'PyTorch',torch.__version__,'GPU',torch.cuda.get_device_name(0))
print('Dataset root',INPUT,'children',[p.name for p in INPUT.iterdir()])


Python 3.13.16 PyTorch 2.11.0+cu130 GPU Tesla T4
Dataset root /kaggle/input/datasets/loliwibu/ripenessmeter-dataset children ['banana_external', 'banana_raw', 'dataset_provenance.json']


## 1 · Source discovery, label contract and license

In [2]:
RAW=INPUT/'banana_raw';EXT=INPUT/'banana_external'
assert RAW.is_dir() and EXT.is_dir(), 'Missing banana_raw or banana_external; check Kaggle Input'
EXTS={'.jpg','.jpeg','.png','.webp'}
raw_files=sorted(p for p in RAW.rglob('*') if p.is_file() and p.suffix.lower() in EXTS)
ext_images=sorted(p for p in (EXT/'images').rglob('*') if p.is_file() and p.suffix.lower() in EXTS)
ext_labels=sorted(p for p in (EXT/'labels').rglob('*.txt'))
print('raw images',len(raw_files),'external images',len(ext_images),'external labels',len(ext_labels))
print('external samples',[str(p.relative_to(EXT)) for p in ext_images[:6]])
assert len(raw_files)==820 and len(ext_images)==673 and len(ext_labels)==673,'Source inventory changed; audit before training'
provenance={'source_primary':'https://data.mendeley.com/datasets/ptfscwtnyz/2','source_external':'https://data.mendeley.com/datasets/56c75hvm7r/1',
            'primary_release':'V2','external_release':'V1 (673 images)','kaggle_mount':str(INPUT),
            'class_semantics':'4-stage original mapped to 3-stage product label; external YOLO classes 0,1,2',
            'external_licence':'CC BY 4.0 reported on source page','primary_licence':'CC BY 4.0 reported on source page',
            'audit_utc':datetime.now(timezone.utc).isoformat()}
(REPORTS/'dataset_provenance.json').write_text(json.dumps(provenance,indent=2),encoding='utf-8')

raw images 820 external images 673 external labels 673
external samples ['images/test/overripe_003.jpg', 'images/test/overripe_013.jpg', 'images/test/overripe_025.jpg', 'images/test/overripe_037.jpg', 'images/test/overripe_040.jpg', 'images/test/overripe_061.jpg']


558

## 2 · Primary image audit, duplicate-conflict quarantine and split

In [3]:
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

def native_name(p):
    s=p.parent.name.casefold().replace('_','').replace(' ','').replace('-','')
    return {'green':'Green','semiripe':'Semi-ripe','ripe':'Ripe','overripe':'Overripe'}[s]

primary=[];bad_primary=[]
for p in raw_files:
    try:
        with Image.open(p) as im:im.verify()
        with Image.open(p) as im:w,h=im.size
        name=native_name(p)
        primary.append({'path':str(p),'source':'BananaImageBD','native_label':name,'label':FOUR_TO_THREE[name],
                        'image_sha256':sha256_file(p),'source_rel':p.relative_to(RAW).as_posix(),'w':w,'h':h})
    except Exception as e:bad_primary.append({'path':str(p),'error':str(e)})
assert not bad_primary,f'Unreadable primary images: {bad_primary[:4]}'
pdf=pd.DataFrame(primary)
# Source-native conflicting labels are quarantined even when 3-class merge would hide conflict.
conflict_keys=pdf.groupby('image_sha256').native_label.nunique()
conflict_hashes=set(conflict_keys[conflict_keys>1].index)
conflicts=pdf[pdf.image_sha256.isin(conflict_hashes)]
conflicts.to_csv(REPORTS/'primary_label_conflicts.csv',index=False)
pdf=pdf[~pdf.image_sha256.isin(conflict_hashes)].copy()
assert pdf.native_label.nunique()==4 and pdf.label.nunique()==3
print('Primary clean',len(pdf),'quarantined',len(conflicts),'native classes',pdf.native_label.value_counts().to_dict())
# Group identical images before splitting.
groups=pdf.groupby('image_sha256').agg(label=('label','first')).reset_index()
assert groups.groupby('image_sha256').label.nunique().max()==1
pg_train,pg_tmp=train_test_split(groups,test_size=.30,random_state=SEED,stratify=groups.label)
pg_val,pg_test=train_test_split(pg_tmp,test_size=.5,random_state=SEED,stratify=pg_tmp.label)
lookup={**{k:'train' for k in pg_train.image_sha256},**{k:'val' for k in pg_val.image_sha256},**{k:'test' for k in pg_test.image_sha256}}
pdf['split']=pdf.image_sha256.map(lookup)
assert pdf.split.notna().all()
print('Primary source split',pd.crosstab(pdf.label,pdf.split))

Primary clean 818 quarantined 2 native classes {'Green': 212, 'Overripe': 203, 'Semi-ripe': 203, 'Ripe': 200}
Primary source split split     test  train  val
label                     
overripe    31    142   30
ripe        30    140   30
unripe      62    290   63


## 3 · External YOLO annotation audit and crop extraction — both datasets now train

In [4]:
# One original image can have multiple object crops. All crops from one image inherit the same split.
# YOLO labels are class_id cx cy w h normalized. Out-of-range boxes are rejected, NOT silently relabeled.
# External original published train/val/test split is honored; crop-level evaluation is grouped by source image.
external_rows=[];invalid_boxes=[];missing_labels=[];image_reports=[]
for idx,p in enumerate(ext_images):
    rel=p.relative_to(EXT/'images')
    split=rel.parts[0].lower() if len(rel.parts)>1 else ''
    assert split in {'train','val','test'},f'Unknown external split: {rel}'
    lab=EXT/'labels'/split/(p.stem+'.txt')
    if not lab.exists():missing_labels.append(str(rel));continue
    try:
        with Image.open(p) as im:
            im=ImageOps.exif_transpose(im).convert('RGB')
            iw,ih=im.size
            valid=0
            for j,line in enumerate(lab.read_text(encoding='utf-8-sig').splitlines()):
                values=line.split()
                if not values:continue
                try:
                    assert len(values)==5
                    class_id=int(values[0]); cx,cy,bw,bh=map(float,values[1:])
                    assert class_id in YOLO_TO_CLASS and all(np.isfinite([cx,cy,bw,bh]))
                    assert 0<=cx<=1 and 0<=cy<=1 and bw>0 and bh>0 and bw<=1 and bh<=1
                    x0=int(round((cx-bw/2)*iw));x1=int(round((cx+bw/2)*iw))
                    y0=int(round((cy-bh/2)*ih));y1=int(round((cy+bh/2)*ih))
                    x0=max(0,x0);x1=min(iw,x1);y0=max(0,y0);y1=min(ih,y1)
                    assert x1-x0>=MIN_BOX_SIDE and y1-y0>=MIN_BOX_SIDE
                except Exception as e:
                    invalid_boxes.append({'image':str(rel),'line':j+1,'raw':line[:140],'reason':str(e)});continue
                if valid>=MAX_CROPS_PER_IMAGE:break
                cr=im.crop((x0,y0,x1,y1))
                # Caching small crops makes repeated 35-epoch training independent of 1.8 GB source decoding.
                dst=CACHE/f'{split}_{idx:04d}_{j:03d}.jpg'
                cr.thumbnail((320,320));cr.save(dst,quality=91)
                external_rows.append({'path':str(dst),'source':'AnnotatedBanana','native_label':YOLO_TO_CLASS[class_id],
                                      'label':YOLO_TO_CLASS[class_id],'split':split,'image_parent':str(rel),
                                      'source_rel':rel.as_posix(),'crop_index':j,'bbox':[x0,y0,x1,y1]})
                valid+=1
            image_reports.append({'image':rel.as_posix(),'accepted_crops':valid,'width':iw,'height':ih})
    except (OSError,ValueError) as e:missing_labels.append(f'{rel}: {e}')
assert not missing_labels, f'Image/label or decode mismatch: {missing_labels[:6]}'
assert external_rows, 'No valid external crops - stop before training'
assert len(set(r['image_parent'] for r in external_rows))>=500, 'Far too few distinct labeled source images after audit'
epd=pd.DataFrame(external_rows)
(REPORTS/'external_box_audit.json').write_text(json.dumps({'images':len(ext_images),'accepted_crops':len(epd),
      'images_with_valid_crops':epd.image_parent.nunique(),'invalid_boxes':invalid_boxes[:300],
      'invalid_boxes_total':len(invalid_boxes),'per_image':image_reports},indent=2),encoding='utf-8')
print('External accepted crops',len(epd),'distinct parents',epd.image_parent.nunique(),
      'rejected bbox lines',len(invalid_boxes))
print('External split',pd.crosstab(epd.label,epd.split))

External accepted crops 662 distinct parents 661 rejected bbox lines 12
External split split     test  train  val
label                     
overripe    24    201   23
ripe        15    167   23
unripe      28    161   20


## 4 · Prevent cross-source exact / near-duplicate contamination; combined manifests

In [5]:
# Unified source manifest and conservative identity screening; run BEFORE loading any training data.
# Exact byte collision and visually near-identical items crossing splits are quarantined.
# Screening uses 64-bit difference-hash + aligned 32x32 RGB normalized MAE; not proof of subject uniqueness.
all_df = pd.concat([pdf, epd], ignore_index=True, sort=False)
all_df['path'] = all_df.path.astype(str)
all_df['file_sha256'] = [sha256_file(Path(s)) for s in all_df.path]
from PIL import ImageFilter

def fingerprint(path):
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert('RGB')
        g = np.asarray(im.resize((9,8), Image.Resampling.BILINEAR).convert('L'), dtype=np.int16)
        bits = (g[:,1:] > g[:,:-1]).reshape(-1)
        h = 0
        for b in bits: h = (h << 1) | int(b)
        thumb = np.asarray(im.resize((32,32), Image.Resampling.BILINEAR),dtype=np.uint8)
    return h, thumb

print('[AUDIT] Fingerprinting', len(all_df), 'eligible files', flush=True)
fp = [fingerprint(Path(x)) for x in all_df.path]
phashes = [x[0] for x in fp]
thumbs = np.stack([x[1] for x in fp])
all_df['dhash64'] = [f'{x:016x}' for x in phashes]
# dHash <= 2 AND 32x32 normalized RGB MAE <= 0.065: conservative near-duplicate candidate.
# JPEG/PNG exact hashing is also checked independently. Similar distinct objects MAY be flagged.
NEAR_HAMMING = 2
NEAR_RGB_MAE = 0.065
candidates=[]; cross_conflicts=set(); exact_pairs=0
# Read-only tuple records avoid millions of expensive DataFrame.iloc calls.
rows=all_df.to_dict('records')
for i in range(len(rows)):
    a=rows[i]
    for j in range(i+1,len(rows)):
        b=rows[j]
        if a['split']==b['split']: continue
        exact=a['file_sha256']==b['file_sha256']
        hd=(phashes[i]^phashes[j]).bit_count()
        if not exact and hd>NEAR_HAMMING:continue
        mae=float(np.abs(thumbs[i].astype(np.int16)-thumbs[j].astype(np.int16)).mean()/255.0) if not exact else 0.
        if exact or mae<=NEAR_RGB_MAE:
            cross_conflicts.update([i,j]);exact_pairs+=int(exact)
            candidates.append({'left':str(a['source_rel']),'left_source':a['source'],'left_split':a['split'],
                'left_label':a['label'],'right':str(b['source_rel']),'right_source':b['source'],
                'right_split':b['split'],'right_label':b['label'],
                'dhash_hamming':int(hd),'rgb_mae':mae,'exact_sha256':bool(exact),
                'same_label':bool(a['label']==b['label'])})
# Also audit exact hash collisions conflicting in class within a split; never trust one label.
bad_label_hashes=set(all_df.groupby('file_sha256').label.nunique().loc[lambda s:s>1].index)
if bad_label_hashes:
    cross_conflicts.update(all_df.index[all_df.file_sha256.isin(bad_label_hashes)].tolist())
reports_dir=REPORTS
pd.DataFrame(candidates,columns=['left','left_source','left_split','left_label','right','right_source',
 'right_split','right_label','dhash_hamming','rgb_mae','exact_sha256','same_label']).to_csv(
 REPORTS/'cross_split_visual_duplicate_candidates.csv',index=False)
quarantine=all_df.iloc[sorted(cross_conflicts)].copy()
quarantine.to_csv(REPORTS/'quarantined_cross_split_samples.csv',index=False)
all_df=all_df.drop(index=sorted(cross_conflicts)).reset_index(drop=True)
assert all_df.groupby('file_sha256').split.nunique().max()==1
assert all_df.groupby('file_sha256').label.nunique().max()==1
assert set(all_df.source)=={'BananaImageBD','AnnotatedBanana'}
assert set(all_df.split)=={'train','val','test'}
assert all_df.groupby(['source','split']).size().min()>=10, 'Too many samples quarantined; review candidates manually'
# A third-party check can reproduce the screening on source data with thresholds in the JSON.
all_df.to_csv(REPORTS/'training_manifest.csv',index=False)
meta={'primary_original_images':820,'primary_conflicted_source_images':int(len(conflicts)),
 'secondary_original_images':673,'secondary_valid_crops_before_screening':len(epd),
 'eligible_samples_after_screening':int(len(all_df)),
 'quarantined_cross_split_samples':int(len(quarantine)),
 'cross_split_visual_duplicate_candidate_pairs':len(candidates),'exact_cross_split_pairs':exact_pairs,
 'screening':{'method':'64-bit dHash Hamming<=2 AND resized RGB mean absolute difference<=0.065 OR exact SHA256',
  'hamming_max':NEAR_HAMMING,'rgb_mae_max':NEAR_RGB_MAE,'limitations':'Screening, NOT guarantee of distinct biological bananas; possible false positives/negatives'},
 'split_counts':{f'{a}_{b}':int(v) for (a,b),v in all_df.groupby(['source','split']).size().items()},
 'external_invalid_box_count':len(invalid_boxes),'source_identity_leakage':'NOT fully verified'}
(REPORTS/'dataset_audit.json').write_text(json.dumps(meta,indent=2),encoding='utf-8')
print('Split after cross-split screening:',pd.crosstab(all_df.source,all_df.split))
print('Near/exact cross-split candidate pairs:',len(candidates),'samples quarantined:',len(quarantine))


[AUDIT] Fingerprinting 1480 eligible files
Split after cross-split screening: split            test  train  val
source                           
AnnotatedBanana    67    527   64
BananaImageBD      98    528  104
Near/exact cross-split candidate pairs: 97 samples quarantined: 92


## 4a · Visual review of suspected cross-split near-duplicates

In [6]:
# Visual gallery of quarantined cross-split pairs for manual label/identity review.
# Do NOT use held-out metrics to choose which pairs to keep.
from PIL import ImageDraw
if len(candidates):
    samples=candidates[:min(8,len(candidates))]
    fig,ax=plt.subplots(len(samples),2,figsize=(9,3*len(samples)),squeeze=False)
    for row,rec in enumerate(samples):
        for col,which in enumerate(['left','right']):
            match=all_df[all_df.source_rel==rec[which]]
            # Paths may be quarantined and no longer in all_df, so recover from original manifest.
            q=quarantine[(quarantine.source_rel==rec[which])&(quarantine.source==rec[which+'_source'])]
            if not len(q):continue
            with Image.open(q.iloc[0].path) as im:ax[row,col].imshow(im.convert('RGB'))
            ax[row,col].set_title(f"{rec[which+'_source']} {rec[which+'_split']} ({rec[which+'_label']})")
            ax[row,col].axis('off')
    fig.tight_layout();fig.savefig(EXAMPLES/'cross_split_candidate_review.png',dpi=110);plt.close(fig)
print('Candidate review visualization saved if candidates exist')


Candidate review visualization saved if candidates exist


## 5 · Auditable image previews and distribution

In [7]:
fig,axes=plt.subplots(2,3,figsize=(11,7))
for row,source in enumerate(['BananaImageBD','AnnotatedBanana']):
    for j,cl in enumerate(CLASSES):
        example=all_df[(all_df.source==source)&(all_df.label==cl)&(all_df.split=='train')].head(1)
        ax=axes[row,j];ax.axis('off');ax.set_title(f'{source} · {cl}')
        if len(example):
            with Image.open(example.iloc[0].path) as im:ax.imshow(im.convert('RGB'))
fig.tight_layout();fig.savefig(EXAMPLES/'two_sources_samples.png',dpi=125);plt.close(fig)
counts=all_df.groupby(['source','label']).size().unstack(fill_value=0).reindex(columns=CLASSES)
ax=counts.T.plot(kind='bar',figsize=(8,4));ax.set_ylabel('Training + validation + test items')
ax.figure.tight_layout();ax.figure.savefig(PLOTS/'two_source_counts.png',dpi=140);plt.close(ax.figure)
print('Saved visual audits')

Saved visual audits


## 6 · Dataset, transforms, domain-balanced sampler

In [8]:
MEAN=(.485,.456,.406);STD=(.229,.224,.225)
tr_tf=transforms.Compose([transforms.Resize((256,256)),transforms.RandomResizedCrop(IMG_SIZE,scale=(.77,1.)),
    transforms.RandomHorizontalFlip(),transforms.ColorJitter(.13,.13,.13,.025),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
va_tf=transforms.Compose([transforms.Resize((256,256)),transforms.CenterCrop(IMG_SIZE),transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
class BananaData(Dataset):
    def __init__(self,frame,aug):self.df=frame.reset_index(drop=True);self.transform=tr_tf if aug else va_tf
    def __len__(self):return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        with Image.open(r.path) as im:image=ImageOps.exif_transpose(im).convert('RGB')
        return self.transform(image),CLASS_TO_ID[r.label]
train=all_df[all_df.split=='train'].copy();val=all_df[all_df.split=='val'].copy();test=all_df[all_df.split=='test'].copy()
# Equal total sampling probability for each dataset source. All eligible samples remain in the dataset.
source_size=train.source.value_counts().to_dict()
weights=[1.0/source_size[s] for s in train.source]
sampler=WeightedRandomSampler(weights,num_samples=len(train),replacement=True,generator=torch.Generator().manual_seed(SEED))
loaders={'train':DataLoader(BananaData(train,True),batch_size=BATCH,sampler=sampler,num_workers=NUM_WORKERS,pin_memory=True),
         'val':DataLoader(BananaData(val,False),batch_size=BATCH,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True),
         'test':DataLoader(BananaData(test,False),batch_size=BATCH,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)}
print({k:len(v.dataset) for k,v in loaders.items()},'train source sizes',source_size)

{'train': 1055, 'val': 168, 'test': 165} train source sizes {'BananaImageBD': 528, 'AnnotatedBanana': 527}


## 7 · Pretrained MobileNetV3-Small, three-stage model

In [9]:
try:
    model=mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
except Exception as exc:
    raise RuntimeError('ImageNet weights unavailable: enable Kaggle Internet or attach a trusted pretrained checkpoint') from exc
model.classifier[-1]=nn.Linear(model.classifier[-1].in_features,3)
model=model.to(DEVICE)
criterion=nn.CrossEntropyLoss(label_smoothing=.03)
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=MAX_EPOCHS)
scaler=torch.amp.GradScaler('cuda')
print('Trainable params',sum(p.numel() for p in model.parameters() if p.requires_grad))

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 108MB/s]


Trainable params 1520931


## 8 · Train with validation-only champion selection

In [10]:
def run_epoch(loader,training):
    model.train(training);tot=0;truth=[];pred=[]
    for x,y in loader:
        x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
        if training:optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.autocast('cuda',enabled=True):logits=model(x);loss=criterion(logits,y)
            if training:
                scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()
        tot+=loss.detach().item()*len(y)
        truth.extend(y.cpu().tolist());pred.extend(logits.argmax(1).detach().cpu().tolist())
    return {'loss':tot/len(loader.dataset),'accuracy':accuracy_score(truth,pred),
            'macro_f1':f1_score(truth,pred,labels=[0,1,2],average='macro',zero_division=0)}
ck=MODELS/'best.pt';best=-1;best_epoch=0;stale=0;history=[];started=time.monotonic()
for epoch in range(1,MAX_EPOCHS+1):
    if epoch>1 and (time.monotonic()-started)/60>MAX_MINUTES:
        print('[TIME CAP] stopping at epoch boundary');break
    tr=run_epoch(loaders['train'],True);va=run_epoch(loaders['val'],False);scheduler.step()
    history.append({'epoch':epoch,**{f'train_{k}':v for k,v in tr.items()},**{f'val_{k}':v for k,v in va.items()}})
    pd.DataFrame(history).to_csv(REPORTS/'training_history.csv',index=False)
    if va['macro_f1']>best+0.001:
        best=va['macro_f1'];best_epoch=epoch;stale=0
        torch.save({'model':model.state_dict(),'classes':CLASSES,'best_epoch':best_epoch,'val_macro_f1':best,
                    'img_size':IMG_SIZE,'sources':['BananaImageBD V2','Annotated Banana V1'],
                    'three_stage_mapping':FOUR_TO_THREE},MODELS/'best.pt.tmp')
        os.replace(MODELS/'best.pt.tmp',ck)
    else:stale+=1
    print(f'Epoch {epoch:02d}/{MAX_EPOCHS} val macro-F1={va["macro_f1"]:.4f} acc={va["accuracy"]:.4f} best={best:.4f} epoch={best_epoch} stale={stale}',flush=True)
    if stale>=PATIENCE:print('[EARLY STOP]');break
assert ck.is_file() and best_epoch>0
print('Selected',best_epoch,'val macro-F1',best)

Epoch 01/35 val macro-F1=0.8015 acc=0.8214 best=0.8015 epoch=1 stale=0
Epoch 02/35 val macro-F1=0.8185 acc=0.8393 best=0.8185 epoch=2 stale=0
Epoch 03/35 val macro-F1=0.8845 acc=0.8869 best=0.8845 epoch=3 stale=0
Epoch 04/35 val macro-F1=0.9231 acc=0.9226 best=0.9231 epoch=4 stale=0
Epoch 05/35 val macro-F1=0.9491 acc=0.9524 best=0.9491 epoch=5 stale=0
Epoch 06/35 val macro-F1=0.9753 acc=0.9762 best=0.9753 epoch=6 stale=0
Epoch 07/35 val macro-F1=0.9770 acc=0.9762 best=0.9770 epoch=7 stale=0
Epoch 08/35 val macro-F1=0.9651 acc=0.9643 best=0.9770 epoch=7 stale=1
Epoch 09/35 val macro-F1=0.9753 acc=0.9762 best=0.9770 epoch=7 stale=2
Epoch 10/35 val macro-F1=0.9708 acc=0.9702 best=0.9770 epoch=7 stale=3
Epoch 11/35 val macro-F1=0.9648 acc=0.9643 best=0.9770 epoch=7 stale=4
Epoch 12/35 val macro-F1=0.9648 acc=0.9643 best=0.9770 epoch=7 stale=5
Epoch 13/35 val macro-F1=0.9712 acc=0.9702 best=0.9770 epoch=7 stale=6
Epoch 14/35 val macro-F1=0.9770 acc=0.9762 best=0.9770 epoch=7 stale=7
[EARLY

## 9 · IMMEDIATE recovery ZIP — save before any test evaluation

In [11]:
def sha256(p):return sha256_file(p)
def zip_result(dest,items):
    temp=dest.with_suffix('.zip.tmp')
    with zipfile.ZipFile(temp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=2,allowZip64=True) as z:
        for p in items:
            if p.is_file():z.write(p,p.relative_to(WORK).as_posix())
    with zipfile.ZipFile(temp) as z:
        assert z.testzip() is None,'CRC failure'
        assert 'models/best.pt' in z.namelist()
    os.replace(temp,dest)
    print('[PASS]',dest.name,round(dest.stat().st_size/2**20,2),'MiB')

(REPORTS/'run_config.json').write_text(json.dumps({'seed':SEED,'architecture':'MobileNetV3-small ImageNet pretrained',
  'classes':CLASSES,'best_epoch':best_epoch,'best_validation_macro_f1':best,
  'image_size':IMG_SIZE,'batch_size':BATCH,'max_epochs':MAX_EPOCHS,
  'checkpoint_sha256':sha256(ck),'selection_rule':'validation macro-F1 only',
  'test_used_for_selection':False},indent=2),encoding='utf-8')
recovery=Path('/kaggle/working/RipenessMeter_Day34_V3_FINAL_RECOVERY.zip')
zip_result(recovery,[ck]+list(REPORTS.glob('*.json'))+list(REPORTS.glob('*.csv')))

[PASS] RipenessMeter_Day34_V3_FINAL_RECOVERY.zip 5.59 MiB


## 10 · Final held-out test + source-specific performance

In [12]:
best_state=torch.load(ck,map_location='cpu',weights_only=True)
model.load_state_dict(best_state['model'],strict=True);model.eval()

def infer_frame(frame):
    ds=BananaData(frame,False);ld=DataLoader(ds,batch_size=BATCH,num_workers=0)
    yy=[];pp=[];confidence=[]
    with torch.inference_mode():
        for x,y in ld:
            with torch.autocast('cuda'):
                z=model(x.to(DEVICE))
            prob=z.float().softmax(1).cpu().numpy()
            yy.extend(y.tolist());pp.extend(prob.argmax(1).tolist());confidence.extend(prob.max(1).tolist())
    metric={'count':len(yy),'accuracy':float(accuracy_score(yy,pp)),
            'macro_f1':float(f1_score(yy,pp,labels=[0,1,2],average='macro',zero_division=0)),
            'balanced_accuracy':float(balanced_accuracy_score(yy,pp)),
            'classification_report':classification_report(yy,pp,labels=[0,1,2],target_names=CLASSES,output_dict=True,zero_division=0)}
    detail=frame.reset_index(drop=True)[['source','label','source_rel','path']].copy()
    detail['predicted']=[CLASSES[i] for i in pp];detail['confidence_softmax_uncalibrated']=confidence
    return metric,detail,confusion_matrix(yy,pp,labels=[0,1,2])
res={};frames={};cms={}
for split_name,frame in [('val',val),('test',test)]:
    for source in ['BananaImageBD','AnnotatedBanana']:
        subset=frame[frame.source==source]
        assert len(subset)>=10,f'{split_name}/{source} too few valid samples'
        met,detail,cm=infer_frame(subset)
        key=f'{split_name}_{source}';res[key]=met;frames[key]=detail;cms[key]=cm
        detail.to_csv(REPORTS/f'{key}_predictions.csv',index=False)
        pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(REPORTS/f'{key}_confusion.csv')
        print(key,'n',met['count'],'acc',round(met['accuracy'],4),'macro-F1',round(met['macro_f1'],4))
(REPORTS/'evaluation_by_source.json').write_text(json.dumps(res,indent=2),encoding='utf-8')

val_BananaImageBD n 104 acc 0.9615 macro-F1 0.9603
val_AnnotatedBanana n 64 acc 1.0 macro-F1 1.0
test_BananaImageBD n 98 acc 0.9388 macro-F1 0.9187
test_AnnotatedBanana n 67 acc 1.0 macro-F1 1.0


3723

## 10a · Parent-image bootstrap confidence intervals

In [13]:
# 95% bootstrap intervals by original-image parent (not naive cropped samples).
# This is uncertainty under the observed collection only, NOT a domain-shift guarantee.
rng=np.random.default_rng(SEED+401)
uncertainty={}
for key,detail in frames.items():
    src=key.split('_',1)[1]
    # Crops within one original AnnotatedBanana image share source_rel; whole primary files are unique.
    grouped=[part for _,part in detail.groupby('source_rel',sort=True)]
    vals=[]
    for _ in range(500):
        idx=rng.integers(0,len(grouped),len(grouped))
        sample=pd.concat([grouped[j] for j in idx],ignore_index=True)
        yt=[CLASS_TO_ID[z] for z in sample['label']]
        yp=[CLASS_TO_ID[z] for z in sample['predicted']]
        vals.append(float(f1_score(yt,yp,labels=[0,1,2],average='macro',zero_division=0)))
    uncertainty[key]={'distinct_parents':len(grouped),'macro_f1_ci95_percentile':
      list(map(float,np.quantile(vals,[.025,.975]))),'method':'500 bootstrap resamples of source image parents'}
(REPORTS/'bootstrap_uncertainty_by_source.json').write_text(json.dumps(uncertainty,indent=2),encoding='utf-8')
print('Uncertainty audit:',json.dumps(uncertainty,indent=2))


Uncertainty audit: {
  "val_BananaImageBD": {
    "distinct_parents": 104,
    "macro_f1_ci95_percentile": [
      0.9218451541960314,
      0.9912038812267645
    ],
    "method": "500 bootstrap resamples of source image parents"
  },
  "val_AnnotatedBanana": {
    "distinct_parents": 64,
    "macro_f1_ci95_percentile": [
      1.0,
      1.0
    ],
    "method": "500 bootstrap resamples of source image parents"
  },
  "test_BananaImageBD": {
    "distinct_parents": 98,
    "macro_f1_ci95_percentile": [
      0.8485895536465973,
      0.9769439875730993
    ],
    "method": "500 bootstrap resamples of source image parents"
  },
  "test_AnnotatedBanana": {
    "distinct_parents": 67,
    "macro_f1_ci95_percentile": [
      1.0,
      1.0
    ],
    "method": "500 bootstrap resamples of source image parents"
  }
}


## 11 · Leakage and uncertainty disclosure, plots, failure cases

In [14]:
hist=pd.read_csv(REPORTS/'training_history.csv')
fig,ax=plt.subplots(figsize=(8,4));ax.plot(hist.epoch,hist.train_macro_f1,label='Train');ax.plot(hist.epoch,hist.val_macro_f1,label='Val')
ax.axvline(best_epoch,ls='--',label='Checkpoint selected');ax.legend();ax.set_title('Macro-F1 by epoch');fig.tight_layout()
fig.savefig(PLOTS/'learning_curve.png',dpi=140);plt.close(fig)
for key,cm in cms.items():
    fig,ax=plt.subplots(figsize=(5,4));ax.imshow(cm,cmap='Blues');ax.set_xticks(range(3),CLASSES);ax.set_yticks(range(3),CLASSES)
    ax.set(xlabel='Prediction',ylabel='Ground truth',title=key)
    for (i,j),n in np.ndenumerate(cm):ax.text(j,i,str(n),ha='center',va='center')
    fig.tight_layout();fig.savefig(PLOTS/f'{key}_confusion.png',dpi=130);plt.close(fig)
# One compact failure grid per source: only 6 images each.
for source in ['BananaImageBD','AnnotatedBanana']:
    frame=frames['test_'+source];wrong=frame[frame.label!=frame.predicted].head(6)
    if wrong.empty:continue
    fig,axes=plt.subplots(2,3,figsize=(11,7));axes=axes.flatten()
    for ax,(_,r) in zip(axes,wrong.iterrows()):
        with Image.open(r.path) as im:ax.imshow(im.convert('RGB'))
        ax.set_title(f'{r.label} → {r.predicted}');ax.axis('off')
    for ax in axes[len(wrong):]:ax.axis('off')
    fig.tight_layout();fig.savefig(EXAMPLES/f'{source}_test_failures.png',dpi=110);plt.close(fig)
print('Saved audit plots')

Saved audit plots


## 12 · Model card + final artifact, no raw images

In [15]:
card=f'''# RipenessMeter Day34 V3 FINAL · Model Card

- Architecture: MobileNetV3-small pretrained; fine-tuned on both BananaImageBD and Annotated Banana V1 crops.
- Target: three classes unripe/ripe/overripe. BananaImageBD Green + Semi-ripe merged into unripe as explicit operational policy.
- Primary source: 820 original whole-image examples minus conflict audit exclusions.
- External source: 673 original YOLO-labeled images yield object crops after strict bbox auditing; original train/val/test membership preserved.
- Champion checkpoint: epoch {best_epoch}, validation macro-F1 {best:.4f}; never selected on test.
- Per-domain evaluation: reports/evaluation_by_source.json.
- Risks: whole-image versus crop domain shift; near-duplicate cross-split candidates quarantined; biological subject identity still not guaranteed; external annotation noise; uncalibrated softmax.
- Usage: single banana image/crop screening, not fruit detection, not autonomous harvest decision.
- Input normalization: ImageNet RGB, resize 256 then center crop 224 at inference.
- Model: PyTorch only. TFLite/ONNX deployment requires conversion and parity verification.
- Source licenses: consult Mendeley dataset V2/V1 before redistribution; raw images not in this ZIP.
'''
(REPORTS/'MODEL_CARD.md').write_text(card,encoding='utf-8')
(REPORTS/'README.md').write_text('RipenessMeter dual-source V3 FINAL: best.pt + manifests + source-separated validation/test results. Never describe 3-stage scores as 4-stage scores.\n',encoding='utf-8')
results=Path('/kaggle/working/RipenessMeter_Day34_V3_FINAL_RESULTS.zip')
(REPORTS/'artifact_manifest.json').write_text(json.dumps({'project':'RipenessMeter Day34 V3 FINAL',
 'checkpoint_sha256':sha256_file(ck),'best_epoch':best_epoch,'best_validation_macro_f1':float(best),
 'test_final_once':True,'quarantine_count':len(quarantine),
 'limitations':['Screened near duplicates are candidates, not verified identities','No external truly novel camera benchmark']},indent=2),encoding='utf-8')
files=[p for p in WORK.rglob('*') if p.is_file() and CACHE not in p.parents]
zip_result(results,files)
print('DONE',recovery,results)

[PASS] RipenessMeter_Day34_V3_FINAL_RESULTS.zip 7.03 MiB
DONE /kaggle/working/RipenessMeter_Day34_V3_FINAL_RECOVERY.zip /kaggle/working/RipenessMeter_Day34_V3_FINAL_RESULTS.zip


## Save Version checklist

1. Kaggle GPU T4, Internet ON on first ImageNet download.
2. Verify `banana_raw` and `banana_external` exist. Invalid YOLO boxes are **reported and omitted**; incompatible source classes never faked.
3. Run `Save Version → Save & Run All`.
4. Download `RipenessMeter_Day34_V2_TWO_DATASETS_RESULTS.zip` plus `RECOVERY.zip` when version reaches **Complete**.
5. Report separate held-out scores **for each source**, not a cherry-picked pooled score. This is still an image/crop-level test with near-duplicate and source mismatch limitations.
6. The published Kaggle package contains only 820 primary raw images and 673 external annotated images. Original variety images/previously augmented BananaImageBD examples were not supplied in this package and cannot be used safely without an updated release and leakage audit.